
# Jupyter Notebook Outline: Voting Network Feature Engineering (Revised)

## 0. Notebook Setup
- **Libraries**: `pandas`, `numpy`, `networkx` / `igraph`, `matplotlib` / `seaborn`, `scikit‑learn`, `stellargraph` or `torch_geometric`, `tqdm`.
- **Data loading**: import raw roll‑call CSVs (votes, bills, lawmakers) and keep date fields for temporal slicing.

## 1. Building Voting Networks
### 1.1 Node Definition  
1. **Deputy‑only graph** – node = deputy; edges weighted by vote similarity (agreement ratio, cosine, etc.).  
2. **Bipartite graph → projection** – deputies + bills; project onto deputies for weighted co‑voting edges.  
3. **Deputy‑Level (Author→Voter) graph** – every bill author (also a deputy) connects to all deputies who cast a vote on that bill.  
   * Directed edges from *author* to *voter* let us study influence and author reach.  
   * Edge weight options:  
     - **Support count**: +1 per “Yes” vote received by the author.  
     - **Signed weight**: +1 for “Yes”, −1 for “No”, 0 for Abstain.  
   * Useful to predict individual deputies’ votes and to analyse how authors are connected to the electorate of peers.

### 1.2 Edge Definition & Weighting  

| Strategy | How to compute | Notes |
|----------|----------------|-------|
| **Simple co‑vote count** | weight = # identical votes | ignores abstentions |
| **Agreement ratio** | weight = agreements / joint votes | 0‑1 scale |
| **Cosine similarity** | treat vote record as vector | handles abstentions |
| **Signed edges** | +1 agreement, −1 disagreement | enables balance analysis |
| **Author‑support** | from author to voter; weight = support count | directed |

### 1.3 Edge Filtering  
- Threshold cut‑off (e.g. weight ≥ 0.6)  
- Backbone extraction (e.g. disparity filter)  
- Temporal decay on older votes  

## 2. Temporal Windows

| Window type | Length | Use‑case |
|-------------|--------|----------|
| Full legislature | 4 years | global portrait |
| Session / year   | 1 year  | evolution across sessions |
| Rolling window   | 3‑, 6‑, 12‑month | short‑term shifts |
| Issue‑based slice| pre/post event | isolate impact |
| Sliding snapshot | fixed width, stepped | dynamic GNNs |

## 3. Feature Engineering from Networks
### 3.1 Node‑level  
- Degree / strength (in‑, out‑ for directed author→voter)  
- Betweenness, closeness, eigenvector, PageRank, Katz, HITS  
- Local clustering coefficient  
- k‑core index  
- Graphlet/role similarity  
- Community ID, community size, internal–external edge ratio  
- **Embeddings**: Node2Vec, DeepWalk, LINE, Struc2Vec; GraphSAGE, GAT, GCN; temporal (TGN, DySAT)  
- Temporal deltas (change in degree, centrality, embedding)

### 3.2 Edge‑level  
- Common neighbours, Jaccard, Adamic‑Adar, Resource Allocation  
- Edge betweenness, signed weight, tie age  

### 3.3 Graph‑level  
- Density, modularity, assortativity, diameter, avg. path length  
- Polarisation index (inter‑community edge ratio)  
- Spectral gap, eigenvalue entropy  

### 3.4 Integration Tips  
1. Combine network features with non‑network covariates.  
2. Scale/normalise continuous metrics; one‑hot categorical ones.  
3. Feature selection or dimensionality reduction when embeddings are many.  
4. Use **time‑aware splits** (train on past, test on future) to avoid leakage.

---

**Notebook flow**  
1. Ingest & clean votes.  
2. Build network variants per chosen window (including Deputy‑Level graph).  
3. Compute node/edge/graph metrics + learned embeddings.  
4. Merge features into a modelling dataframe.  
5. Train baseline classifiers (logistic regression, random forest, GNN) with time‑aware validation.  
6. Iterate: adjust thresholds, add temporal deltas, compare performance.


## 1. Data Ingetion

In [1]:
import pandas as pd